### Set autoreloading
This extension will automatically update with any changes to packages in real time

In [1]:
#%load_ext autoreload
#%autoreload 2

### Import packages
We'll need the `pytorch_lightning` and `nugraph` packages imported in order to run inference, the `time` package to measure inference time, and `plotly.graph_objects` to plot.

In [2]:
import time
import pytorch_lightning as pl
import nugraph as ng
import plotly.graph_objects as go

/home/csaia/miniforge3/envs/numl/lib/python3.12/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/csaia/nugraph/nugraph/nugraph/util/scriptutils.py:2: FutureWarning: The pynvml_utils module is deprecated. The source will remain accessible on Github at https://github.com/gpuopenanalytics/pynvml/.
  from pynvml_utils import nvidia_smi


### Set default plotting options

Define a dictionary containing all standard plotting options that we want to set for all the plots we draw

In [3]:
style = {
    'layout_width': 800,
    'layout_height': 450,
    'layout_margin_b': 20,
    'layout_margin_t': 20,
    'layout_margin_r': 20,
    'layout_margin_l': 20,
    'layout_xaxis_title_font_size': 24,
    'layout_xaxis_tickfont_size': 20,
    'layout_yaxis_title_font_size': 24,
    'layout_yaxis_tickfont_size': 20,
    'layout_legend_font_size': 24,
}

### Configure network
In order to test a trained model, we instantiate it using a checkpoint file. These are produced during training, so if you've trained a model, there should be an associated checkpoint in your output directory that you can pass here.

In [5]:
model = ng.models.NuGraph2.load_from_checkpoint('/home/csaia/nugraph/logs/icarus_numi_test/version_0/checkpoints/epoch=63-step=4032.ckpt', map_location='cpu')
model.freeze()

NuGraph2(
  (encoder): Encoder(
    (net): ModuleDict(
      (u): Sequential(
        (0): ClassLinear(
          (net): ModuleList(
            (0-4): 5 x Linear(in_features=4, out_features=64, bias=True)
          )
        )
        (1): Tanh()
      )
      (v): Sequential(
        (0): ClassLinear(
          (net): ModuleList(
            (0-4): 5 x Linear(in_features=4, out_features=64, bias=True)
          )
        )
        (1): Tanh()
      )
      (y): Sequential(
        (0): ClassLinear(
          (net): ModuleList(
            (0-4): 5 x Linear(in_features=4, out_features=64, bias=True)
          )
        )
        (1): Tanh()
      )
    )
  )
  (plane_net): PlaneNet(
    (net): ModuleDict(
      (u): MessagePassing2D()
      (v): MessagePassing2D()
      (y): MessagePassing2D()
    )
  )
  (nexus_net): NexusNet(
    (nexus_up): SimpleConv()
    (nexus_net): Sequential(
      (0): ClassLinear(
        (net): ModuleList(
          (0-4): 5 x Linear(in_features=192, out_f

### Benchmark inference time
Loop over a range of batch size options. For each one, run testing and record how long it took.

In [ ]:
accelerator, devices = ng.util.configure_device()
x = []
y = []
for i in range(9):
    batch_size = pow(2, i)
    x.append(batch_size)
    nudata = ng.data.H5DataModule(
        data_path='/exp/icarus/data/users/csaia/NuGraphH5-BNB-merge/numl_v1_00.gnn.h5', #'/raid/uboone/NuGraph2/NG2-paper.gnn.h5',
        batch_size=batch_size,
    )
    accelerator, devices = ng.util.configure_device()
    trainer = pl.Trainer(accelerator=accelerator,
                         devices=devices, logger=False)
    t0 = time.time()
    trainer.test(model, datamodule=nudata)
    y.append((time.time()-t0)/len(nudata.test_dataset))

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        loss/test           1.0368841886520386
    loss_filter/test        0.5232967138290405
   loss_semantic/test       0.5135875940322876
     memory_cpu/test        0.6666297912597656
  precision_filter/test     0.8951690196990967
 precision_semantic/test     0.705788791179657
   recall_filter/test       0.9843325614929199
  recall_semantic/test       0.705788791179657
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Testing: |          | 0/? [00:00<?, ?it/s]

...

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        loss/test           0.8315001726150513
    loss_filter/test        0.7075384855270386
   loss_semantic/test       0.12396165728569031
     memory_cpu/test        0.9954910278320312
  precision_filter/test     0.8560426235198975
 precision_semantic/test    0.7210118174552917
   recall_filter/test       0.9842766523361206
  recall_semantic/test      0.7210118174552917
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


Testing: |          | 0/? [00:00<?, ?it/s]

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        loss/test           0.8279293775558472
    loss_filter/test        0.7129009962081909
   loss_semantic/test       0.11502843350172043
     memory_cpu/test        0.9954910278320312
  precision_filter/test     0.8553152084350586
 precision_semantic/test     0.721161961555481
   recall_filter/test       0.9841256141662598
  recall_semantic/test       0.721161961555481
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


Testing: |          | 0/? [00:00<?, ?it/s]

### Plot inference time
Draw a scatter plot using the batch sizes and inference times from the previous step. Save the resulting plot to disk and visualize it.

In [ ]:
fig = go.Figure(
    layout_xaxis_title='Batch size',
    layout_yaxis_title='Inference time per graph [s]',
    layout_xaxis_type='log',
    layout_xaxis_tickmode='array',
    layout_xaxis_ticktext=x,
    layout_xaxis_tickvals=x,
    **style,
)
fig.add_scatter(x=x, y=y)

fig.write_image('plots/inference-time.png')
fig.write_image('plots/inference-time.pdf')

fig